In [27]:
# Homework 3 _ Feature Group

In [ ]:
#%pip install --force-reinstall 'sagemaker>=2.200,<3'
import boto3
import sagemaker
from sagemaker.session import Session
from sagemaker import get_execution_role
import pandas as pd
import numpy as np
import time
from time import gmtime, strftime, sleep
from sagemaker.feature_store.feature_group import FeatureGroup
import pprint



sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [3]:
print("sagemaker version:", sagemaker.__version__)  # expect something like 2.254.x

sagemaker version: 2.257.3


In [4]:
original_boto3_version = boto3.__version__
#%pip install -q 'boto3>1.17.21'

In [5]:
region = boto3.Session().region_name
boto_session = boto3.Session(region_name=region)

sagemaker_client     = boto_session.client("sagemaker", region_name=region)
featurestore_runtime = boto_session.client("sagemaker-featurestore-runtime", region_name=region)

feature_store_session = Session(
    boto_session=boto_session,
    sagemaker_client=sagemaker_client,
    sagemaker_featurestore_runtime_client=featurestore_runtime,
)

default_s3_bucket_name = feature_store_session.default_bucket()
prefix = "sagemaker-featurestore-homework-3-1"
print("Region :", region)
print("Bucket :", default_s3_bucket_name)

Region : us-east-1
Bucket : sagemaker-us-east-1-697214847177


In [6]:
role = get_execution_role()
print("Role   :", role)

Role   : arn:aws:iam::697214847177:role/LabRole


In [7]:
HOUSING_PATH = "housing.csv"
GMAPS_PATH   = "housing_gmaps_data_raw.csv"

housing = pd.read_csv(HOUSING_PATH)
gmaps   = pd.read_csv(GMAPS_PATH)

print("housing:", housing.shape, "  gmaps:", gmaps.shape)
housing.head()

housing: (20640, 10)   gmaps: (12590, 30)


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [8]:
gmaps_slim = (gmaps[["latitude", "longitude", "neighborhood-political", "postal_code"]]
              .drop_duplicates(subset=["latitude", "longitude"]))

df = housing.merge(gmaps_slim, on=["latitude", "longitude"], how="left")

# neighborhood label
df = df[df["neighborhood-political"].notna()].copy()

print("Rows after join + neighborhood filter:", len(df))
print("Distinct neighborhoods               :", df['neighborhood-political'].nunique())

Rows after join + neighborhood filter: 9000
Distinct neighborhoods               : 1306


In [9]:
# one-hot encode ocean_proximity with all 5 categories enforced
ALL_OCEAN = ["<1H OCEAN", "INLAND", "ISLAND", "NEAR BAY", "NEAR OCEAN"]
df["ocean_proximity"] = pd.Categorical(df["ocean_proximity"], categories=ALL_OCEAN)

ocean_oh = pd.get_dummies(df["ocean_proximity"]).astype(int)
ocean_oh.columns = (ocean_oh.columns.str.lower()
                                    .str.replace("<1h ocean", "lt_1h_ocean", regex=False)
                                    .str.replace(" ", "_"))
df = pd.concat([df, ocean_oh], axis=1)

OH_COLS = list(ocean_oh.columns)
print("One-hot columns:", OH_COLS)

One-hot columns: ['lt_1h_ocean', 'inland', 'island', 'near_bay', 'near_ocean']


In [10]:
# bedrooms_per_household at the row level
df["bedrooms_per_household"] = df["total_bedrooms"] / df["households"]

# Impute: where the ratio is NaN (because total_bedrooms was null),
# fill with the mean ratio for that row's postal code
df["bedrooms_per_household"] = (df.groupby("postal_code")["bedrooms_per_household"]
                                  .transform(lambda s: s.fillna(s.mean())))

# Safety net: a postal code with zero non-null ratios would leave residual NaNs
df["bedrooms_per_household"] = df["bedrooms_per_household"].fillna(
    df["bedrooms_per_household"].mean()
)

print("Residual nulls in bedrooms_per_household:", df["bedrooms_per_household"].isna().sum())

Residual nulls in bedrooms_per_household: 0


In [11]:
agg = (df.groupby("neighborhood-political")
         .agg(**{c: (c, "max") for c in OH_COLS},
              median_house_value   = ("median_house_value", "mean"),
              median_house_age_raw = ("housing_median_age", "mean"),
              total_households     = ("households",         "mean"),
              bedrooms_per_household = ("bedrooms_per_household", "mean"))
         .reset_index()
         .rename(columns={"neighborhood-political": "neighborhood"}))

# Cap median_house_value at 500,000
agg["median_house_value"] = agg["median_house_value"].clip(upper=500000)

# Round households up to integer
agg["total_households"] = np.ceil(agg["total_households"]).astype(int)

# Bucket the averaged median house age into 10year ranges
def age_bucket(avg_age):
    lo = int(avg_age // 10) * 10
    return f"{lo}-{lo+9}"

agg["median_house_age"] = agg["median_house_age_raw"].apply(age_bucket)
agg = agg.drop(columns=["median_house_age_raw"])

# Final column order
neighborhood_features = agg[
    ["neighborhood"] + OH_COLS +
    ["median_house_value", "median_house_age", "total_households", "bedrooms_per_household"]
].copy()

print("Feature group shape:", neighborhood_features.shape)
print("Any nulls?", neighborhood_features.isna().any().any())
neighborhood_features.head()

Feature group shape: (1306, 10)
Any nulls? False


,neighborhood,lt_1h_ocean,inland,island,near_bay,near_ocean,median_house_value,median_house_age,total_households,bedrooms_per_household
0,28 Palms,1,0,0,0,0,222200.000000,20-29,923,1.017335
1,Acorn Industrial,0,0,0,1,0,81300.000000,50-59,147,1.659864
2,Adams Hill,1,0,0,0,0,250733.333333,30-39,494,1.034649
3,Agua Mansa Industrial Corridor,0,1,0,0,0,112300.000000,10-19,516,1.102713
4,Al Tahoe,0,1,0,0,0,109180.000000,20-29,249,1.641739


In [12]:
#Building the Feature Group

In [13]:
def cast_object_to_string(frame):
    for col in frame.columns:
        if frame.dtypes[col] == "object":
            frame[col] = frame[col].astype("str").astype("string")

cast_object_to_string(neighborhood_features)

record_identifier_feature_name = "neighborhood"
event_time_feature_name        = "event_time"

current_time_sec = float(round(time.time()))
neighborhood_features[event_time_feature_name] = pd.Series(
    [current_time_sec] * len(neighborhood_features), dtype="float64"
)

neighborhood_features.dtypes

neighborhood                  str
lt_1h_ocean                 int64
inland                      int64
island                      int64
near_bay                    int64
near_ocean                  int64
median_house_value        float64
median_house_age              str
total_households            int64
bedrooms_per_household    float64
event_time                float64
dtype: object

In [16]:
for col in ["neighborhood", "median_house_age"]:
    neighborhood_features[col] = neighborhood_features[col].astype(object)

print(neighborhood_features.dtypes)  # verify

neighborhood               object
lt_1h_ocean                 int64
inland                      int64
island                      int64
near_bay                    int64
near_ocean                  int64
median_house_value        float64
median_house_age           object
total_households            int64
bedrooms_per_household    float64
event_time                float64
dtype: object


In [17]:
neighborhood_feature_group_name = "neighborhood-feature-group-" + strftime("%d-%H-%M-%S", gmtime())
print("FeatureGroup name:", neighborhood_feature_group_name)

neighborhood_feature_group = FeatureGroup(
    name=neighborhood_feature_group_name,
    sagemaker_session=feature_store_session,
)

neighborhood_feature_group.load_feature_definitions(data_frame=neighborhood_features)

FeatureGroup name: neighborhood-feature-group-25-14-30-23


[FeatureDefinition(feature_name='neighborhood', feature_type=<FeatureTypeEnum.STRING: 'String'>, collection_type=None),
 FeatureDefinition(feature_name='lt_1h_ocean', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None),
 FeatureDefinition(feature_name='inland', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None),
 FeatureDefinition(feature_name='island', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None),
 FeatureDefinition(feature_name='near_bay', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None),
 FeatureDefinition(feature_name='near_ocean', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None),
 FeatureDefinition(feature_name='median_house_value', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='median_house_age', feature_type=<FeatureTypeEnum.STRING: 'String'>, collection_type=None),
 FeatureDefinition(fe

In [18]:
def wait_for_feature_group_creation_complete(fg):
    status = fg.describe().get("FeatureGroupStatus")
    while status == "Creating":
        print("Waiting for FeatureGroup creation...")
        time.sleep(5)
        status = fg.describe().get("FeatureGroupStatus")
    if status != "Created":
        raise RuntimeError(f"Failed to create FeatureGroup {fg.name}")
    print(f"FeatureGroup {fg.name} successfully created.")

neighborhood_feature_group.create(
    s3_uri=f"s3://{default_s3_bucket_name}/{prefix}",
    record_identifier_name=record_identifier_feature_name,
    event_time_feature_name=event_time_feature_name,
    role_arn=role,
    enable_online_store=True,
)

wait_for_feature_group_creation_complete(neighborhood_feature_group)

Waiting for FeatureGroup creation...


Waiting for FeatureGroup creation...


Waiting for FeatureGroup creation...


Waiting for FeatureGroup creation...


FeatureGroup neighborhood-feature-group-25-14-30-23 successfully created.


In [19]:
neighborhood_feature_group.describe()

{'FeatureGroupArn': 'arn:aws:sagemaker:us-east-1:697214847177:feature-group/neighborhood-feature-group-25-14-30-23',
 'FeatureGroupName': 'neighborhood-feature-group-25-14-30-23',
 'RecordIdentifierFeatureName': 'neighborhood',
 'EventTimeFeatureName': 'event_time',
 'FeatureDefinitions': [{'FeatureName': 'neighborhood',
   'FeatureType': 'String'},
  {'FeatureName': 'lt_1h_ocean', 'FeatureType': 'Integral'},
  {'FeatureName': 'inland', 'FeatureType': 'Integral'},
  {'FeatureName': 'island', 'FeatureType': 'Integral'},
  {'FeatureName': 'near_bay', 'FeatureType': 'Integral'},
  {'FeatureName': 'near_ocean', 'FeatureType': 'Integral'},
  {'FeatureName': 'median_house_value', 'FeatureType': 'Fractional'},
  {'FeatureName': 'median_house_age', 'FeatureType': 'String'},
  {'FeatureName': 'total_households', 'FeatureType': 'Integral'},
  {'FeatureName': 'bedrooms_per_household', 'FeatureType': 'Fractional'},
  {'FeatureName': 'event_time', 'FeatureType': 'Fractional'}],
 'CreationTime': dat

In [20]:
neighborhood_feature_group.ingest(
    data_frame=neighborhood_features,
    max_workers=3,
    wait=True,
)

IngestionManagerPandas(feature_group_name='neighborhood-feature-group-25-14-30-23', feature_definitions={'neighborhood': {'FeatureName': 'neighborhood', 'FeatureType': 'String'}, 'lt_1h_ocean': {'FeatureName': 'lt_1h_ocean', 'FeatureType': 'Integral'}, 'inland': {'FeatureName': 'inland', 'FeatureType': 'Integral'}, 'island': {'FeatureName': 'island', 'FeatureType': 'Integral'}, 'near_bay': {'FeatureName': 'near_bay', 'FeatureType': 'Integral'}, 'near_ocean': {'FeatureName': 'near_ocean', 'FeatureType': 'Integral'}, 'median_house_value': {'FeatureName': 'median_house_value', 'FeatureType': 'Fractional'}, 'median_house_age': {'FeatureName': 'median_house_age', 'FeatureType': 'String'}, 'total_households': {'FeatureName': 'total_households', 'FeatureType': 'Integral'}, 'bedrooms_per_household': {'FeatureName': 'bedrooms_per_household', 'FeatureType': 'Fractional'}, 'event_time': {'FeatureName': 'event_time', 'FeatureType': 'Fractional'}}, sagemaker_fs_runtime_client_config=<botocore.confi

In [21]:
def fetch(neighborhood_name):
    resp = featurestore_runtime.get_record(
        FeatureGroupName=neighborhood_feature_group_name,
        RecordIdentifierValueAsString=neighborhood_name,
    )
    print(f"=== {neighborhood_name} ===")
    pprint.pp(resp["Record"])
    print()
    return resp

_ = fetch("Brooktree")

=== Brooktree ===
[{'FeatureName': 'neighborhood', 'ValueAsString': 'Brooktree'},
 {'FeatureName': 'lt_1h_ocean', 'ValueAsString': '1'},
 {'FeatureName': 'inland', 'ValueAsString': '0'},
 {'FeatureName': 'island', 'ValueAsString': '0'},
 {'FeatureName': 'near_bay', 'ValueAsString': '0'},
 {'FeatureName': 'near_ocean', 'ValueAsString': '0'},
 {'FeatureName': 'median_house_value', 'ValueAsString': '257400.0'},
 {'FeatureName': 'median_house_age', 'ValueAsString': '0-9'},
 {'FeatureName': 'total_households', 'ValueAsString': '1438'},
 {'FeatureName': 'bedrooms_per_household',
  'ValueAsString': '1.0747777761942723'},
 {'FeatureName': 'event_time', 'ValueAsString': '1779718688.0'}]



In [22]:
_ = fetch("Fisherman's Wharf")

=== Fisherman's Wharf ===
[{'FeatureName': 'neighborhood', 'ValueAsString': "Fisherman's Wharf"},
 {'FeatureName': 'lt_1h_ocean', 'ValueAsString': '0'},
 {'FeatureName': 'inland', 'ValueAsString': '0'},
 {'FeatureName': 'island', 'ValueAsString': '0'},
 {'FeatureName': 'near_bay', 'ValueAsString': '1'},
 {'FeatureName': 'near_ocean', 'ValueAsString': '0'},
 {'FeatureName': 'median_house_value', 'ValueAsString': '500000.0'},
 {'FeatureName': 'median_house_age', 'ValueAsString': '50-59'},
 {'FeatureName': 'total_households', 'ValueAsString': '250'},
 {'FeatureName': 'bedrooms_per_household', 'ValueAsString': '1.268'},
 {'FeatureName': 'event_time', 'ValueAsString': '1779718688.0'}]



In [23]:
_ = fetch("Los Osos")

=== Los Osos ===
[{'FeatureName': 'neighborhood', 'ValueAsString': 'Los Osos'},
 {'FeatureName': 'lt_1h_ocean', 'ValueAsString': '0'},
 {'FeatureName': 'inland', 'ValueAsString': '0'},
 {'FeatureName': 'island', 'ValueAsString': '0'},
 {'FeatureName': 'near_bay', 'ValueAsString': '0'},
 {'FeatureName': 'near_ocean', 'ValueAsString': '1'},
 {'FeatureName': 'median_house_value', 'ValueAsString': '221612.5'},
 {'FeatureName': 'median_house_age', 'ValueAsString': '10-19'},
 {'FeatureName': 'total_households', 'ValueAsString': '612'},
 {'FeatureName': 'bedrooms_per_household',
  'ValueAsString': '1.0478845404823531'},
 {'FeatureName': 'event_time', 'ValueAsString': '1779718688.0'}]



In [24]:
batch = featurestore_runtime.batch_get_record(Identifiers=[{
    "FeatureGroupName": neighborhood_feature_group_name,
    "RecordIdentifiersValueAsString": ["Brooktree", "Fisherman's Wharf", "Los Osos"],
}])

rows = []
for rec in batch["Records"]:
    rows.append({f["FeatureName"]: f["ValueAsString"] for f in rec["Record"]})

pd.DataFrame(rows).set_index("neighborhood")

,lt_1h_ocean,inland,island,near_bay,near_ocean,median_house_value,median_house_age,total_households,bedrooms_per_household,event_time
neighborhood,,,,,,,,,,
Brooktree,1,0,0,0,0,257400.0,0-9,1438,1.0747777761942723,1779718688.0
Fisherman's Wharf,0,0,0,1,0,500000.0,50-59,250,1.268,1779718688.0
Los Osos,0,0,0,0,1,221612.5,10-19,612,1.0478845404823531,1779718688.0
